In [109]:
import os
os.chdir("..")
import pandas as pd
import numpy as np
import torch
from PIL import Image
from data.dataset import CropDataset

In [76]:
DATA_DIR = ""

In [131]:
DATASET_DIR = ""

In [132]:
DATASET_PATH = os.path.join(DATASET_DIR, "")

In [78]:
df = pd.read_csv(DATASET_PATH)

/tmp/mirgau/ipykernel_147651/2461292460.py:1: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET_PATH)


In [118]:
maize_df = df[df["crop"]=="Maize/Corn"]
rice_df = df[df["crop"]=="Rice"]

In [119]:
nigeria_2022_df = rice_df[(rice_df['data_collection_year'] == 2022) & (rice_df['country'] == 'Nigeria')]
zambia_2023_df = maize_df[(maize_df['data_collection_year'] == 2023) & (maize_df['country'] == 'Zambia')]
zambia_2024_df = maize_df[(maize_df['data_collection_year'] == 2024) & (maize_df['country'] == 'Zambia')]
zimbabwe_2024_df = maize_df[(maize_df['data_collection_year'] == 2024) & (maize_df['country'] == 'Zimbabwe')]

In [120]:
t = CropDataset.build_transform(True, 224)  # or define this once outside
def is_image_valid(cce_id):
    path = os.path.join(DATA_DIR, f"q_field_photo/q_field_photo_{cce_id}.jpg")
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            t(img)
        return True
    except Exception:
        return False

def filter_corrupted_images(df):
    valid_flags = [is_image_valid(cce_id) for cce_id in df["cce_id"]]
    return df[valid_flags].reset_index(drop=True)

In [121]:
def filter_nan(df):
    return df.dropna(subset=["Avg_yield_mt_ha"]).reset_index(drop=True)

In [122]:
def add_zone_column(df):
    df["zone"] = df["data_collection_year"].astype(str) + '_' + df["AEZ"].astype(str) + '_' + df["Client"].astype(str)
    return df

In [123]:
def filter_small_zones(df):
    zone_counts = df["zone"].value_counts()
    valid_zones = zone_counts[zone_counts >= 20].index
    return df[df["zone"].isin(valid_zones)].reset_index(drop=True)

In [124]:
def create_folds(df, num_folds=5):
    df = df.copy()
    df['fold'] = -1

    for zone in df["zone"].unique():
        indices = df[df["zone"] == zone].index
        shuffled_indices = np.random.permutation(indices)

        # Split into folds
        split_indices = np.array_split(shuffled_indices, num_folds)

        # Shuffle the fold numbers to randomize which folds get the extra samples
        fold_order = np.random.permutation(num_folds)

        for fold, fold_indices in zip(fold_order, split_indices):
            df.loc[fold_indices, 'fold'] = fold

    return df.reset_index(drop=True)

In [125]:
nigeria_2022_filtered_corrupt_df=filter_corrupted_images(nigeria_2022_df)
nigeria_2022_filtered_nan_df = filter_nan(nigeria_2022_filtered_corrupt_df)
nigeria_2022_zone_df = add_zone_column(nigeria_2022_filtered_nan_df)
nigeria_2022_zone_filter_df = filter_small_zones(nigeria_2022_zone_df)
nigeria_2022_fold_df = create_folds(nigeria_2022_zone_filter_df)
nigeria_2022_fold_df.to_csv(os.path.join(DATASET_DIR,"nigeria_2022.csv"), index=False)

In [126]:
zambia_2023_filtered_corrupt_df=filter_corrupted_images(zambia_2023_df)
zambia_2023_filtered_nan_df = filter_nan(zambia_2023_filtered_corrupt_df)
zambia_2023_zone_df = add_zone_column(zambia_2023_filtered_nan_df)
zambia_2023_zone_filter_df = filter_small_zones(zambia_2023_zone_df)
zambia_2023_fold_df = create_folds(zambia_2023_zone_filter_df)
zambia_2023_fold_df.to_csv(os.path.join(DATASET_DIR,"zambia_2023.csv"), index=False)

In [127]:
zambia_2024_filtered_corrupt_df=filter_corrupted_images(zambia_2024_df)
zambia_2024_filtered_nan_df = filter_nan(zambia_2024_filtered_corrupt_df)
zambia_2024_zone_df = add_zone_column(zambia_2024_filtered_nan_df)
zambia_2024_zone_filter_df = filter_small_zones(zambia_2024_zone_df)
zambia_2024_fold_df = create_folds(zambia_2024_zone_filter_df)
zambia_2024_fold_df.to_csv(os.path.join(DATASET_DIR,"zambia_2024.csv"), index=False)

In [128]:
zimbabwe_2024_filtered_corrupt_df=filter_corrupted_images(zimbabwe_2024_df)
zimbabwe_2024_filtered_nan_df = filter_nan(zimbabwe_2024_filtered_corrupt_df)
zimbabwe_2024_zone_df = add_zone_column(zimbabwe_2024_filtered_nan_df)
zimbabwe_2024_zone_filter_df = filter_small_zones(zimbabwe_2024_zone_df)
zimbabwe_2024_fold_df = create_folds(zimbabwe_2024_zone_filter_df)
zimbabwe_2024_fold_df.to_csv(os.path.join(DATASET_DIR,"zimbabwe_2024.csv"), index=False)